# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [4]:
col = "Amount_invested_monthly"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad < Mediana Standard < Mediana Good.

Además los deudores buenos tiene una cola más pesada.

Esto concuerda con la hipótesis de que los deudores malos invierten menos y los buenos, más.

In [5]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [6]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [7]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Amount_invested_monthly_Decile,,,,,,,,,,
0,0.000000,44.139440,10000,0.1,3626,1696,4678,0.3626,0.1696,0.4678
1,44.144394,64.125471,10000,0.1,3138,2135,4727,0.3138,0.2135,0.4727
2,64.126298,83.586181,10000,0.1,2825,2452,4723,0.2825,0.2452,0.4723
3,83.586757,105.264808,10000,0.1,2808,2659,4533,0.2808,0.2659,0.4533
4,105.266810,129.360954,10000,0.1,2579,2774,4647,0.2579,0.2774,0.4647
5,129.362723,160.573129,10000,0.1,2294,3027,4679,0.2294,0.3027,0.4679
6,160.577867,203.608349,10000,0.1,2188,3171,4641,0.2188,0.3171,0.4641
7,203.608418,275.964606,10000,0.1,1780,3585,4635,0.1780,0.3585,0.4635
8,275.978796,426.443725,10000,0.1,1635,3971,4394,0.1635,0.3971,0.4394


No missing values found.
No infinite values found.
No duplicate rows found.


In [8]:
df[(df[continuous_variable] >= 0.000000) & (df[continuous_variable] <= 44.139440) & (df['Credit_Score'] == 0)].shape

(3626, 92)

Proporción de Buenos: se observa una relación positiva, a mayor el monto invertido mayor la proporción de buenos deudores.

Proporción de standard: la realación entre el monto invertido y la proporción de deudores Standard es negativa, en tendencia.

Proporción de malos: según lo esperado, la proporción de malos y la magnitud de las inversiones se relacionan negativamente.

In [9]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

Dado el comportamiento monótono por decil, no se proponen agrupaciones de estos.


## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [10]:
df_ = df_deciles.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [11]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Amount_invested_monthly,"100,000.00",193.69,194.79,0.00,73.73,129.36,234.35,"1,977.33"


Todos los coeficientes son significativos.

Amount_invested_monthly_Scaled 4.4419: el coeficiente es positivo: por cada dólar adicional invertido, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.7743: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7261: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [12]:
df_ = df_deciles.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.036007
         Iterations: 11
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0360e+05
Model:                   OrderedModel   AIC:                         2.072e+05
Method:            Maximum Likelihood   BIC:                         2.072e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:56:31                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                     coef

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Amount_invested_monthly_Decile 0.1446: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.5591: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7290: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [13]:
df_ = df_deciles.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.036904
         Iterations: 9
         Function evaluations: 11
         Gradient evaluations: 11
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0369e+05
Model:                   OrderedModel   AIC:                         2.074e+05
Method:            Maximum Likelihood   BIC:                         2.074e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:59:11                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                     coef 

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Amount_invested_monthly` usando Regresión Ordinal

| Representación                         | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Amount_invested_monthly_Scaled`       | 4.4419                | **Log-Likelihood**: -103,600<br>**AIC**: 207,201<br>**BIC**: 207,221                   | 🔹 Mejor ajuste global.<br>🔹 Alta pendiente indica una fuerte relación positiva con el `Credit_Score`. |
| `Amount_invested_monthly_Decile`       | 0.1446                | **Log-Likelihood**: -103,690<br>**AIC**: 207,381<br>**BIC**: 207,402                   | 🔹 Ligera pérdida de ajuste por discretización.<br>🔹 Relación positiva, pero menos pronunciada. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [ ]:
# df_deciles.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
